# Шаги 3–4. Сопоставление и метрики предпосылки (CPU)


In [ ]:
!pip install -q pymorphy3
!git clone -q --depth 1 https://github.com/vladislavneon/RuBQ.git /kaggle/working/RuBQ || true

In [ ]:
%%writefile match.py
"""
match_strict(a, b, kind) и match_lemma(a, b, kind) применяются без изменений
и к парам (ответ, эталон/алиас), и к парам (сэмпл_i, сэмпл_j).
kind — тип эталона вопроса: 'entity' | 'string' | 'number' | 'date'.
"""
import re
import unicodedata
from collections import Counter

def clean(s):
    s = s.split("\n")[0]                                  
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")  # ударения: «Но́вая» → «Новая»
    s = unicodedata.normalize("NFC", s).lower().replace("ё", "е")
    s = re.sub(r"[^\w\s]", " ", s)                         # пунктуация → пробел
    return re.sub(r"\s+", " ", s).strip()

MONTHS = {  # основа месяца → номер; ловит «февраль», «февраля», «феврале»
    "январ": 1, "феврал": 2, "март": 3, "апрел": 4, "ма": 5, "июн": 6,
    "июл": 7, "август": 8, "сентябр": 9, "октябр": 10, "ноябр": 11, "декабр": 12,
}
MONTH_RX = re.compile(r"\b(январ\w*|феврал\w*|март\w*|апрел\w*|ма[йяе]|июн\w*|июл\w*|"
                      r"август\w*|сентябр\w*|октябр\w*|ноябр\w*|декабр\w*)\b")

def canon_number(s):
    s = s.split("\n")[0].lower().replace(",", ".")
    s = re.sub(r"(?<=\d)[\s ](?=\d{3}\b)", "", s)     # «1 500» → «1500»
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    if not m:
        return None
    x = float(m.group())
    return int(x) if x == int(x) else x

def canon_date(s):
    """→ (год, месяц, день); отсутствующие компоненты = None."""
    s = s.split("\n")[0].lower()
    iso = re.match(r"^\s*(-?\d{1,4})-(\d{2})-(\d{2})t", s)
    if iso:
        return int(iso.group(1)), int(iso.group(2)), int(iso.group(3))
    year = re.search(r"\b(\d{3,4})\b", s)
    mon = MONTH_RX.search(s)
    month = None
    if mon:
        w = mon.group(1)
        month = next(v for k, v in MONTHS.items() if w.startswith(k))
    day = re.search(r"\b(\d{1,2})\b", s) if mon else None
    return (int(year.group(1)) if year else None, month, int(day.group(1)) if day else None)

def match_date(a, b):
    da, db = canon_date(a), canon_date(b)
    if da[0] is None or db[0] is None:
        return False
    return all(x is None or y is None or x == y for x, y in zip(da, db))

def match_number(a, b):
    na, nb = canon_number(a), canon_number(b)
    return na is not None and na == nb

# --- строгое ---
def match_strict(a, b, kind="entity"):
    if kind == "number":
        return match_number(a, b)
    if kind == "date":
        return match_date(a, b)
    ca, cb = clean(a), clean(b)
    return bool(ca) and ca == cb

# --- лемматизированное ---
STOP = {  # предлоги и союзы — фиксированный список
    "в", "во", "на", "из", "с", "со", "у", "о", "об", "обо", "по", "за", "под", "при", "от",
    "до", "к", "ко", "для", "через", "над", "перед", "между", "около", "без", "про",
    "и", "или", "а", "но", "да",
}
_morph = None
def _lemma(w):
    global _morph
    if _morph is None:
        import pymorphy3
        _morph = pymorphy3.MorphAnalyzer()
    return _morph.parse(w)[0].normal_form.replace("ё", "е")

def lemmas(s):
    return Counter(_lemma(t) for t in clean(s).split() if t not in STOP)

def match_lemma(a, b, kind="entity"):
    if kind in ("number", "date"):
        return match_strict(a, b, kind)       # числа/даты: то же правило, лемматизация не влияет
    la, lb = lemmas(a), lemmas(b)
    return bool(la) and la == lb

# --- эталон вопроса ---
DT = "http://www.w3.org/2001/XMLSchema#"
def q_kind(q):
    a = q["answers"][0]
    if a["type"] == "uri":
        return "entity"
    dt = a.get("datatype", "")
    if dt == DT + "dateTime":
        return "date"
    if dt in (DT + "integer", DT + "decimal"):
        return "number"
    return "string"

def gold_strings(q):
    """label/value + wd_names.ru + wd_names.en + wp_names по всем эталонным ответам."""
    out = []
    for a in q["answers"]:
        out += [a.get("label") or a["value"]]
        out += a["wd_names"]["ru"] + a["wd_names"]["en"] + a["wp_names"]
    return [s for s in dict.fromkeys(out) if s]

def correct(answer, q, fn):
    k = q_kind(q)
    return any(fn(answer, g, k) for g in gold_strings(q))


# Шаг 4. Метрики предпосылки. 
Использует match.py без изменений.

In [ ]:
%%writefile metrics.py
import json, random, itertools, glob
from collections import Counter
import numpy as np
from match import match_strict, match_lemma, q_kind, gold_strings, lemmas, clean

def load(gen_path, rubq_path):
    rubq = {q["uid"]: q for q in json.load(open(rubq_path))}
    rows = [json.loads(l) for l in open(gen_path)]
    return rows, rubq

def correct(ans, q, fn):
    k = q_kind(q)
    return any(fn(ans, g, k) for g in gold_strings(q))

def group_sc(samples, kind, fn):
    """Жадная группировка: сэмпл идёт в первую группу, с представителем которой совпадает."""
    reps, sizes = [], []
    for s in samples:
        for i, r in enumerate(reps):
            if fn(s, r, kind):
                sizes[i] += 1
                break
        else:
            reps.append(s); sizes.append(1)
    return max(sizes) / len(samples)

def per_question(rows, rubq):
    out = []
    for r in rows:
        q = rubq[r["uid"]]; k = q_kind(q); S = r["samples"]
        pairs_l = pairs_ls = 0
        merged = []
        for i, j in itertools.combinations(range(len(S)), 2):
            l = match_lemma(S[i], S[j], k)
            if l:
                pairs_l += 1
                if not match_strict(S[i], S[j], k):
                    pairs_ls += 1; merged.append((S[i], S[j]))
        out.append(dict(
            uid=r["uid"], kind=k, question=r["question"], greedy=r["greedy"],
            gold=q["answers"][0].get("label") or q["answers"][0]["value"],
            ok_s=correct(r["greedy"], q, match_strict), ok_l=correct(r["greedy"], q, match_lemma),
            pairs_l=pairs_l, pairs_ls=pairs_ls, merged=merged,
            sc_s=group_sc(S, k, match_strict), sc_l=group_sc(S, k, match_lemma),
            mean_lp=float(np.mean(r["greedy_logprobs"])) if r["greedy_logprobs"] else float("nan"),
        ))
    return out

def stats(P):
    ok_s = np.array([p["ok_s"] for p in P]); ok_l = np.array([p["ok_l"] for p in P])
    any_ok = ok_s | ok_l; diff = ok_s != ok_l
    pl = np.array([p["pairs_l"] for p in P]); pls = np.array([p["pairs_ls"] for p in P])
    dsc = np.array([p["sc_l"] - p["sc_s"] for p in P])
    return {
        "Acc_strict": ok_s.mean(), "Acc_lemma": ok_l.mean(),
        "D1": diff.sum() / max(any_ok.sum(), 1),
        "D2": pls.sum() / max(pl.sum(), 1),
        "D2_sc": (np.abs(dsc) > 1e-9).mean(), "mean_abs_dSC": np.abs(dsc).mean(),
    }

def counts(P):
    ok_s = np.array([p["ok_s"] for p in P]); ok_l = np.array([p["ok_l"] for p in P])
    return {"n": len(P), "D1_abs": int((ok_s != ok_l).sum()), "any_ok": int((ok_s | ok_l).sum()),
            "strict_only": int((ok_s & ~ok_l).sum()), "lemma_only": int((ok_l & ~ok_s).sum()),
            "pairs_lemma": int(sum(p["pairs_l"] for p in P)),
            "pairs_lemma_not_strict": int(sum(p["pairs_ls"] for p in P)),
            "q_sc_changed": int(sum(abs(p["sc_l"] - p["sc_s"]) > 1e-9 for p in P))}

def bootstrap(P, B=1000, seed=42):
    rng = np.random.default_rng(seed); n = len(P)
    boot = [stats([P[i] for i in rng.integers(0, n, n)]) for _ in range(B)]
    return {k: (float(np.percentile([b[k] for b in boot], 2.5)),
                float(np.percentile([b[k] for b in boot], 97.5))) for k in boot[0]}

def decision(st, ci):
    """Правило решения; при пересечении порога интервалом берём более осторожный вариант (по нижним границам)."""
    if st["Acc_lemma"] < 0.15:
        return "Acc_lemma < 15%: switch to one larger model, repeat steps 2-4, no decision", None, False
    def rule(a, b):
        if a >= 0.10 or b >= 0.10: return "Premise strong: proceed with the thesis plan as written"
        if a < 0.03 and b < 0.03: return "Mechanism does not work: reformulate the topic or switch to the backup topic"
        return "Proceed with emphasis on H2 (sample grouping) and native data"
    point = rule(st["D1"], st["D2_sc"])
    cautious = rule(ci["D1"][0], ci["D2_sc"][0])
    # граница: 95% CI хотя бы одной метрики содержит порог 3% или 10%
    border = any(lo < t < hi for t in (0.03, 0.10) for (lo, hi) in (ci["D1"], ci["D2_sc"]))
    return point, cautious, border

def manual_sample(P, n=50, seed=42):
    """50 случайных случаев: D1-спасения (ответ vs эталон) + D2-слияния (сэмпл vs сэмпл)."""
    cases = []
    for p in P:
        if p["ok_l"] and not p["ok_s"]:
            cases.append(dict(src="D1", uid=p["uid"], question=p["question"], a=p["greedy"], b=p["gold"]))
        for a, b in dict.fromkeys(p["merged"]):   # уникальные пары строк
            cases.append(dict(src="D2", uid=p["uid"], question=p["question"], a=a, b=b))
    random.Random(seed).shuffle(cases)
    for c in cases:
        c["lemmas_a"] = " ".join(sorted(lemmas(c["a"]).elements()))
        c["lemmas_b"] = " ".join(sorted(lemmas(c["b"]).elements()))
        c["false_merge"] = ""   # заполнить вручную: 1 = разные сущности слиплись, 0 = корректно
    return cases[:n], len(cases)


In [ ]:
# Шаг 3: проверка лемматизации на контрольных парах (до подсчёта)
from match import match_strict, match_lemma, lemmas
for a, b in [("в Москве", "Москва"), ("Толстого", "Толстой"), ("Франции", "Франция"),
             ("Вирджинии Вулф", "Вирджиния Вулф"), ("у Пушкина", "Пушкин"), ("классицизм", "Классицизм"),
             ("Россия", "Росси"), ("Германия", "Германии Восточной")]:
    print(f"{a!r:22} {b!r:24} strict={match_strict(a, b)!s:5} lemma={match_lemma(a, b)!s:5} {dict(lemmas(a))} {dict(lemmas(b))}")

In [ ]:
# Шаг 4: метрики
import glob, json
import numpy as np
from metrics import *
GEN = (glob.glob("/kaggle/input/**/generations.jsonl", recursive=True)
       + glob.glob("/kaggle/working/generations.jsonl"))
print(GEN); assert GEN, "generations.jsonl не найден ни в /kaggle/input, ни в /kaggle/working"
rows, rubq = load(GEN[0], "/kaggle/working/RuBQ/RuBQ_2.0/RuBQ_2.0_test.json")
assert len(rows) == 300, len(rows)
P = per_question(rows, rubq)
st, cn, ci = stats(P), counts(P), bootstrap(P, B=1000, seed=42)
print(f"{'метрика':14} {'значение':>9}   95% CI")
for k in st:
    print(f"{k:14} {st[k]:9.3f}   [{ci[k][0]:.3f}, {ci[k][1]:.3f}]")
print(cn)
point, cautious, border = decision(st, ci)
print("\nРешение по точечным оценкам:", point)
print("По нижним границам CI:       ", cautious, "| граница пересечена:", border)
json.dump({"stats": {k: float(v) for k, v in st.items()}, "ci": ci, "counts": cn,
           "decision_point": point, "decision_cautious": cautious, "border": border},
          open("/kaggle/working/metrics.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# Примеры расхождений
from collections import Counter
print(Counter(p["kind"] for p in P))
print("\nD1 — lemma спасла ответ:")
for p in [p for p in P if p["ok_l"] != p["ok_s"]][:10]:
    print(f"  [{p['kind']}] {p['question'][:70]} | эталон: {p['gold']} | ответ: {p['greedy']!r} | strict={p['ok_s']} lemma={p['ok_l']}")
print("\nD2 — слияния сэмплов:")
for p in [p for p in P if p["merged"]][:10]:
    print(f"  {p['question'][:60]} | {p['merged'][0]} | SC strict={p['sc_s']:.1f} lemma={p['sc_l']:.1f}")

In [ ]:
# Выборка 50 случаев для ручной проверки ложных слияний
import csv
cases, total = manual_sample(P, n=50, seed=42)
print("всего кандидатов:", total, "| в выборке:", len(cases))
with open("/kaggle/working/manual_check.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(cases[0])); w.writeheader(); w.writerows(cases)
for c in cases:
    print(f"{c['src']} | {c['a']!r} ~ {c['b']!r} | {c['lemmas_a']} / {c['lemmas_b']}")